# Test Notebook

The purpose of this notebook is to test the functions created in `preprocessing.py`, `modelling.py`, and `evaluation.py`. 

Eventually this notebook will serve as a how to set up the top of your notebook in a way that enables you to access/use all of the functions in the scripts contained in the src folder.

This notebook should also give an explanation about the .vscode + .json file that was created to allow easy access to the src folder. Since .vscode is one of the ignored folders in .gitignore, it does not get pushed to the repo. Most likely you need to push to repo and integrate .vscode into the main branch once (after taking it off the .gitignore), and then ignore any further changes to the .vscode folder. Will need to test this with the team. 

### For the README
The `src/` folder contains python (`.py`) scripts needed for running the `.ipynb` files contained in the `notebooks/` folder. The functions in each script help with a certain aspect of the project. At the top of each `.ipynb` in the `notebooks/` folder you will find a code block that sets up access to the `src/` folder and its functions. If you remove this code block you will not be able to properly run the notebook. 

### For the team
- I added a hidden folder called .vscode that has a single .json file that tells VS Code to automatically add the `src` folder to the path
- I don't think you will need to do manually add this on your machines once it has been integrated with the main branch.
- you may not need to manually add it before pushing it to main, either. 

#### Functions
Many of the functions are well documented. See function docstrings for more info on how to use each function.

**preprocessing.py**: functions for data ingestion and preprocessing 
- `load_raw_data()`: data ingestion, standardized DataFrame setup
- `get_XY()`
- `split_data()`
- `prepare_model_data()`

In [9]:
#----------------------------------
# Setup: Import functions from src
#----------------------------------
import sys
from pathlib import Path

# temporary...fix pyproject.toml later (pin the version of sklearn)
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Get repo root (one level above notebooks/)
repo_root = Path().resolve().parent
src_path = repo_root / "src"

# Add src folder to Python path
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

# import modules
    # prepare_model_data() produces ML pipeline ready preprocessed data 
    # Functions can be standalone, but not generally advised. 
    # To avoid potential data leakage only use prepare_model_data(). 
    # One exception: load_raw_data for EDA/figures that require label access
from preprocessing import (
    load_raw_data, get_Xy, split_data, prepare_model_data
)

from modelling import logistic_pipeline, rf_pipeline, svm_pipeline
from evaluation import evaluate_model, compare_models

print(f"All functions from src/preprocessing.py are ready to use")
print(f"All functions from src/modelling.py are ready to use")
print(f"All functions from src/evaluation.py are ready to use")


All functions from src/preprocessing.py are ready to use
All functions from src/modelling.py are ready to use
All functions from src/evaluation.py are ready to use


In [10]:
import os

# the code below may need to be changed if we put anything else in raw other than the wdbc data
SRC_PATH = "../data/raw/"
for root, dirs, files in os.walk(SRC_PATH):
    for file in files:
        if file.endswith(".data") or file.endswith(".csv"):
            data_path = os.path.join(root, file)

In [11]:
# load raw data, clean and return
df = load_raw_data(data_path)
df.head()

,diagnosis,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,malignant,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,malignant,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,malignant,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,malignant,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,malignant,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [12]:
# separate data into X, y
X, y = get_Xy(df)

print(f'Features: {X.shape[1]}, Samples: {X.shape[0]}')

Features: 30, Samples: 569


In [13]:
# split data
X_train, X_test, y_train, y_test = split_data(X, y)

print(f'Train: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')
print(f'Train class distribution:\n{y_train.value_counts()}')
print(f'Test class distribution:\n{y_test.value_counts()}')

Train: 455 samples
Test:  114 samples
Train class distribution:
diagnosis
1    285
0    170
Name: count, dtype: int64
Test class distribution:
diagnosis
1    72
0    42
Name: count, dtype: int64


In [14]:
# prepare_model_data() produces ML pipeline ready preprocessed data 
data = prepare_model_data(data_path)

# SCALING HAPPENS IN THE PIPELINES

print(f'Train: {data.X_train.shape[0]} samples')
print(f'Test:  {data.X_test.shape[0]} samples')
print(f'Train class distribution:\n{data.y_train.value_counts()}')
print(f'Test class distribution:\n{data.y_test.value_counts()}')

Train: 455 samples
Test:  114 samples
Train class distribution:
diagnosis
1    285
0    170
Name: count, dtype: int64
Test class distribution:
diagnosis
1    72
0    42
Name: count, dtype: int64


In [15]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# run a model using the team workflow
models = {
    "Log_base": logistic_pipeline(penalty=None),
    "Log_L1": logistic_pipeline(penalty="l1"),
    "Log_L2": logistic_pipeline(penalty="l2"),
    "RF": rf_pipeline(),
    "SVM": svm_pipeline()
}

results = compare_models(models, data.X_train, data.y_train, data.X_test, data.y_test)

In [16]:
print(results)

   accuracy  precision    recall        f1   roc_auc     model
0  0.921053   0.970149  0.902778  0.935252  0.971892  Log_base
1  0.991228   0.986301  1.000000  0.993103  0.996693    Log_L1
2  0.982456   0.986111  0.986111  0.986111  0.995370    Log_L2
3  0.956140   0.958904  0.972222  0.965517  0.993056        RF
4  0.982456   0.986111  0.986111  0.986111  0.995040       SVM
